# R7-A1 - Brazo del agente de optimización: etapa 1

**Objetivo.** Aplicar de forma literal las reglas escritas de antemano en `results/classification/r7_optimizacion/agente/protocolo_agente.json` (con la enmienda E1) a las 45 corridas de la etapa 1: nueve candidatas sobre ConvNeXt-Tiny con imagen completa, cinco folds de E2 y semilla 42, frente a las referencias B0 y B2 (corridas existentes de `robustez_e2`). El notebook elige el finalista con `seleccion_del_finalista` y genera, solo si lo hay, los 30 comandos de la confirmación. No añade reglas ni usa test.

**Diseño.** Cada candidata cambia un solo factor respecto de B0; no hay combinaciones. Todas conservan el criterio de checkpoint (máximo `val_roc_auc`), α = 1.0 y el early stopping por pérdida ponderada de validation.

| Candidata | Factor | Flags |
|---|---|---|
| `a01_lr5e5`, `a02_lr2e4` | learning rate pico | `--learning-rate 5e-5` / `2e-4` |
| `a03_cosine` | calentamiento y coseno | `--lr-schedule warmup_cosine --warmup-epochs 1 --cosine-epochs 15 --lr-min 1e-6` |
| `a04_llrd08` | decaimiento por capa | `--layer-decay 0.8` |
| `a05_dp03` | stochastic depth | `--drop-path-rate 0.3` |
| `a06_ls01` | label smoothing uniforme (E1) | `--label-smoothing 0.1` |
| `a08_augmod` | intensidad del aumento | `--augmentation moderate` |
| `a09_res288` | resolución | `--image-size 288` |
| `a10_ema099` | EMA de pesos | `--ema-decay 0.99` |

E2 es estratificada y no por paciente, y el AP del finalista es el máximo de nueve estimaciones ruidosas: está sesgado al alza. La estimación válida del desempeño es la de test en R8.

## 1. Configuración

In [ ]:
from pathlib import Path
import json
import shlex
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def locate_project_root() -> Path:
    current_dir = Path.cwd().resolve()

    for candidate in [current_dir, *current_dir.parents]:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate

    raise FileNotFoundError("No se pudo localizar la raíz del proyecto.")


PROJECT_ROOT = locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.classification import agente_etapa1_analysis as a1  # toda la lógica del análisis está en este módulo

AGENT_DIR = PROJECT_ROOT / a1.AGENT_REL
ANALYSIS_DIR = AGENT_DIR / a1.ANALYSIS_NAME
OUTPUTS = {key: ANALYSIS_DIR / name for key, name in a1.OUTPUT_FILES.items()}

PROTOCOL, PROTOCOL_PATH = a1.load_agent_protocol(PROJECT_ROOT)
CANDIDATES = a1.candidate_table(PROTOCOL)  # en el orden del protocolo
RUNS = a1.stage1_runs(PROTOCOL)
CONFIGS = list(CANDIDATES) + ["b0", "b2"]
N_FOLDS = a1.N_FOLDS

COLORS = {"b0": "#0072B2", "b2": "#009E73"}
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})


def require(condition, message):
    if not condition:
        raise AssertionError(message)


print(f"Protocolo: {PROTOCOL_PATH.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Estado declarado: {PROTOCOL['estado']}")
print(f"Enmiendas: {[e['id'] for e in PROTOCOL['enmiendas']]} | candidatas: {len(CANDIDATES)} | corridas de la etapa 1: {len(RUNS)}")

## 2. Protocolo y comandos de entrenamiento

Las 45 corridas y su comando están escritos en el protocolo; aquí solo se indica cuáles ya tienen resultados. Con `RUN_TRAINING = False` no se inicia ningún entrenamiento. Con `True` (solo en Falcon) se lanzan las que falten, sin sobrescribir carpetas con archivos. Una corrida que falla por infraestructura se repite una vez con el mismo comando; si vuelve a fallar, la candidata se registra como descartada y cuenta como probada.

In [ ]:
RUN_TRAINING = False  # True para lanzar desde este notebook las corridas que falten (Falcon); False solo las lista

done = [run["id"] for run in RUNS if (PROJECT_ROOT / run["dir_rel"] / "validation_metrics.json").exists()]
print(f"Corridas con resultados: {len(done)} de {len(RUNS)}")
print("Ejemplo de comando:\n" + RUNS[0]["command"])

if not RUN_TRAINING:
    print("\nRUN_TRAINING = False: no se inicia ningún entrenamiento.")
else:
    for run in RUNS:
        output_dir = PROJECT_ROOT / run["dir_rel"]
        print("=" * 80)
        print(f"{run['id']} -> {run['dir_rel']}/")

        if output_dir.exists() and any(output_dir.iterdir()):
            print("Omitida: la carpeta ya contiene archivos y no se sobrescribe. Si una corrida quedó incompleta, revísela o muévala manualmente.")
            continue

        command = [sys.executable, *shlex.split(run["command"])[1:]]
        process = subprocess.Popen(
            command, cwd=PROJECT_ROOT,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors="replace", bufsize=1,
        )
        try:
            for line in process.stdout:
                print(line, end="")
        except KeyboardInterrupt:
            process.terminate()  # al interrumpir la celda no debe quedar un entrenamiento huérfano
            raise
        if process.wait() != 0:
            raise RuntimeError(f"La corrida {run['id']} terminó con error (código {process.returncode}).")

## 3. Carga y verificación de las corridas

Esta sección solo lee archivos. Para cada corrida comprueba que la configuración registrada coincide con el comando del protocolo (semilla, flags, hiperparámetros fijos, criterio de checkpoint, early stopping), que el manifiesto del fold no incluye test y que su validation es el fold de `cv_folds_e2.csv`, que los pesos de clase corresponden al fold con α = 1.0, y que AP y ROC-AUC se reproducen desde las probabilidades guardadas y desde el historial. Aplica las mismas comprobaciones a B0 y B2, y comprueba que sus valores coinciden con los escritos en el protocolo. Si una comprobación falla o falta alguna carpeta, no se escribe ningún resultado. Una carpeta incompleta se registra como fallida y no entra en la selección.

Las tablas se escriben en `agente/analisis_etapa1/` mediante `src/classification/agente_etapa1_analysis.py`; si ya existen, solo se leen.

In [ ]:
missing_runs = [run["dir_rel"] for run in RUNS if not (PROJECT_ROOT / run["dir_rel"]).exists()]
require(not missing_runs, "Faltan carpetas de la etapa 1 (git pull o completar las corridas):\n" + "\n".join(missing_runs))

existing_outputs = [path for key, path in OUTPUTS.items() if path.exists()]
if not existing_outputs:
    exit_code, _ = a1.run_analysis(PROJECT_ROOT, verbose=False)
    require(exit_code == 0, "El análisis no se escribió: hay comprobaciones fallidas (ejecute `python -m src.classification.agente_etapa1_analysis`).")
else:
    required = [path for key, path in OUTPUTS.items() if key != "commands"]  # los comandos solo existen si hay finalista
    require(all(path.exists() for path in required), "analisis_etapa1/ está incompleta; revise la carpeta antes de continuar.")

verification = pd.read_csv(OUTPUTS["verification"])
print(f"Comprobaciones: {len(verification)} | fallidas: {int((~verification['ok']).sum())}")
display(verification.groupby("comprobacion", sort=False)["ok"].agg(corridas="size", todas_ok="all"))

## 4. AP y ROC-AUC por candidata y fold

AP y ROC-AUC del checkpoint de cada corrida (máximo `val_roc_auc` del fold), con semilla 42. No dependen del umbral. Cada fold tiene 21 o 22 positivas operacionales: el AP por fold es ruidoso y la media no es una estimación final. El registro completo de intentos, incluidas las corridas fallidas, está en `registro_intentos.csv`.

In [ ]:
metrics = pd.read_csv(OUTPUTS["metrics"])

for column, title in (("ap", "AP"), ("roc_auc", "ROC-AUC")):
    table = metrics.pivot(index="config_id", columns="fold", values=column).reindex([c for c in CONFIGS if c in set(metrics["config_id"])])
    table["media"] = table.mean(axis=1)
    print(title)
    display(table.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
ordered = [c for c in CONFIGS if c in set(metrics["config_id"])]
for position, config in enumerate(ordered):
    values = metrics.loc[metrics["config_id"] == config, "ap"]
    ax.scatter([position] * len(values), values, color=COLORS.get(config, "0.35"), s=30, zorder=3)
    ax.hlines(values.mean(), position - 0.3, position + 0.3, color="black", lw=1.5, zorder=4)
for reference in ("b0", "b2"):
    ax.axhline(metrics.loc[metrics["config_id"] == reference, "ap"].mean(), color=COLORS[reference], ls="--", lw=1, label=f"{reference.upper()} (media)")
ax.set_xticks(range(len(ordered)))
ax.set_xticklabels(ordered, rotation=45, ha="right")
ax.set_ylabel("Average Precision (validation)")
ax.set_title("Figura 1. AP por fold de cada candidata y referencia (semilla 42); la línea negra es la media")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(ANALYSIS_DIR / "fig1_ap_por_fold.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Comparaciones frente a B0 y B2

El delta por fold es la candidata menos la referencia, con las mismas imágenes de validation. Se informan media, mínimo, máximo, desviación estándar muestral (solo descriptiva) y signos; un delta exactamente 0 no cuenta en ningún signo. La consistencia (mismo signo estricto en al menos 4 de 5 folds) se evalúa en ambas direcciones y solo con el AP; la ROC-AUC se reporta de forma descriptiva. Con cinco folds no se reporta p-valor ni intervalo de confianza. Se incluye B2 frente a B0 como referencia.

In [ ]:
summary = pd.read_csv(OUTPUTS["delta_summary"])
ap_summary = summary[summary["metrica"] == "AP"].rename(columns={"folds_positivos": "folds +", "folds_negativos": "folds -", "folds_cero": "folds 0"})
display(ap_summary[["comparacion", "media", "minimo", "maximo", "folds +", "folds -", "folds 0", "veredicto"]].round(4).set_index("comparacion"))
auc_summary = summary[summary["metrica"] == "ROC-AUC"].set_index("comparacion")
print("ΔROC-AUC medio frente a B0 (descriptivo):")
display(auc_summary.loc[[c for c in auc_summary.index if c.endswith("__vs_b0")], ["media", "minimo", "maximo"]].round(4))

In [ ]:
vs_b0 = ap_summary[ap_summary["comparacion"].str.endswith("__vs_b0")].copy()
vs_b0["candidata"] = vs_b0["comparacion"].str.replace("__vs_b0", "", regex=False)
fig, ax = plt.subplots(figsize=(9, 4.5))
positions = np.arange(len(vs_b0))
ax.errorbar(positions, vs_b0["media"], yerr=[vs_b0["media"] - vs_b0["minimo"], vs_b0["maximo"] - vs_b0["media"]],
            fmt="o", color="#0072B2", ecolor="0.5", capsize=4, label="media y rango (mín–máx) de ΔAP por fold")
ax.axhline(0, color="black", lw=1)
ax.set_xticks(positions)
ax.set_xticklabels(vs_b0["candidata"], rotation=45, ha="right")
ax.set_ylabel("Δ AP frente a B0")
ax.set_title("Figura 2. Diferencia de AP frente a B0 por candidata")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(ANALYSIS_DIR / "fig2_delta_ap_vs_b0.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Selección del finalista

Aplicación mecánica de `seleccion_del_finalista`: son elegibles las candidatas con ΔROC-AUC medio frente a B0 mayor o igual que -0.02; el finalista es la elegible con mayor AP medio entre los 5 folds; si hay empate (diferencia menor que 0.005 respecto del máximo) decide la mayor ROC-AUC media y luego el menor id. Una candidata con algún fold fallido queda descartada y cuenta como probada. Si ninguna es elegible no hay confirmación ni ganadora y se aplica `sin_ganadora`.

In [ ]:
selection = json.loads(OUTPUTS["selection"].read_text(encoding="utf-8"))
print(f"Protocolo idéntico al commit {selection['protocolo']['commit_de_referencia']}: {selection['protocolo']['idéntico_al_commit']}")
print(f"Corridas: {selection['conteo']['completadas']} completadas, {selection['conteo']['fallidas']} fallidas | umbrales: {selection['regla']['umbrales_leidos_del_texto']}")

table = pd.DataFrame(selection["candidatas"]).set_index("id")
display(table[["ap_medio", "roc_auc_medio", "delta_auc_medio_vs_b0", "elegible", "descartada", "motivo"]].round(4))
print(f"Elegibles: {selection['elegibles']}")
print(f"AP medio máximo: {selection['ap_maximo']} | en empate: {selection['en_empate']}")
print(f"Finalista: {selection['finalista']} | el desempate cambia el finalista: {selection['el_desempate_cambia_el_finalista']}")
print(f"Consecuencia: {selection['consecuencia']}")

## 7. Comandos de confirmación

Solo si hay finalista: 30 comandos (finalista, B0 y B2; semillas 43 y 44; cinco folds) generados por sustitución mecánica de la plantilla del protocolo y verificados contra ella. La decisión usa solo las semillas 43 y 44. El autor debe comprobar que coinciden con la plantilla antes de correr.

In [ ]:
if not OUTPUTS["commands"].exists():
    print("No hay finalista: no se genera comandos_confirmacion.json (75 -> 45 corridas).")
else:
    payload = json.loads(OUTPUTS["commands"].read_text(encoding="utf-8"))
    print(f"Finalista: {payload['finalista']} | flags: {payload['flags_del_finalista']} | comandos: {payload['total']}")
    display(pd.Series(payload["verificacion_contra_la_plantilla"], name="cumple"))
    print(payload["comandos"][0]["comando"])

## 8. Limitaciones

- E2 es estratificada y no por paciente (no hay identificadores de paciente): puede sobreestimar, y los cinco folds no son independientes entre sí porque comparten imágenes de entrenamiento.
- Una semilla por fold en la etapa 1: no se separa la variabilidad por semilla de la variabilidad por fold. Cada fold de validation tiene 21 o 22 positivas operacionales, de modo que el AP por fold es ruidoso; con una semilla, el error estándar de ΔAP es del orden del margen de 0.04 de la regla de elección (cálculo propio del protocolo, por verificar).
- El AP medio del finalista es el máximo de nueve estimaciones ruidosas: sesgo al alza (selección del ganador). Con la regla de signo de 4 de 5 folds y nueve candidatas, la probabilidad de que alguna la cumpla por azar frente a B0 es de ≈ 0.85 bajo la hipótesis nula del protocolo, un cálculo aproximado (los folds no son independientes).
- Las 1000 imágenes de E2 incluyen las 200 de validation de E1 con las que se seleccionaron B2 y el criterio de checkpoint: E2 no es independiente de esa selección.
- El efecto de cada factor se mide solo desde la base B0 y sin combinaciones: no se detectan interacciones. El early stopping por pérdida ponderada puede detener el entrenamiento antes del máximo de AUC; no se modifica porque el criterio está fijo.
- Las categorías 6 y 7 están agrupadas en la clase positiva operacional; la categoría 6 no es malignidad histológica. Varias fuentes del protocolo están marcadas [CITA POR VERIFICAR].
- Solo se usan archivos de validation; test no interviene. La estimación válida del desempeño es la de test en R8.

## 9. Interpretación de resultados

*Pendiente.* Esta sección se redacta solo después de ejecutar el notebook, con las cifras de los archivos de `agente/analisis_etapa1/` y sin añadir reglas a las del protocolo.